In [ ]:
# Example 1 — transformers.pipelines: the pre-process -> forward -> post-process chain users call.
#
# The other cells construct configs and models and call them with tensors already in hand. A
# `pipeline` is the layer above that, and it is a different failure surface: it selects a task class,
# runs the tokenizer, batches and pads, moves tensors to the device, calls the model, then decodes
# logits back into labelled dicts. A break in that chain — a task registry that lost its entry, a
# postprocessor returning raw logits, a tokenizer/model handshake that silently truncates — leaves
# every `BertModel(**inputs)` assertion elsewhere green while every user-facing call returns nonsense.
# Built ENTIRELY from in-memory objects: no hub, no network, no cached weights.
import numpy as np
from tokenizers import Tokenizer, models, pre_tokenizers
from transformers import BertConfig, BertForSequenceClassification, PreTrainedTokenizerFast, pipeline

vocab = {"[UNK]": 0, "[PAD]": 1, "[CLS]": 2, "[SEP]": 3, "good": 4, "bad": 5, "very": 6}
backend = Tokenizer(models.WordLevel(vocab, unk_token="[UNK]"))
backend.pre_tokenizer = pre_tokenizers.Whitespace()
tokenizer = PreTrainedTokenizerFast(tokenizer_object=backend, unk_token="[UNK]", pad_token="[PAD]",
                                    cls_token="[CLS]", sep_token="[SEP]")
config = BertConfig(vocab_size=len(vocab), hidden_size=32, num_hidden_layers=1,
                    num_attention_heads=2, intermediate_size=32, num_labels=2,
                    id2label={0: "NEG", 1: "POS"}, label2id={"NEG": 0, "POS": 1})
model = BertForSequenceClassification(config).eval()

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device="cpu")
out = classifier(["very good", "very bad"])
assert len(out) == 2, out
assert {r["label"] for r in out} <= {"NEG", "POS"}, out          # id2label really reached the output
assert all(0.0 <= r["score"] <= 1.0 for r in out), out           # softmax ran, not raw logits
assert np.isclose(sum(r["score"] for r in classifier("very good", top_k=None)), 1.0), "not normalized"
assert classifier("very good")[0] == classifier("very good")[0]  # deterministic in eval mode
print("transformers pipeline: task resolved, tokenizer+model+postprocess ->", out[0]["label"])

In [ ]:
# Example 2 — the core model runtime: build a tiny BertModel locally (random
# weights, NO network) and run one real forward pass. Exercises nn.Module init,
# the encoder stack, and attention on CPU.
import torch
from transformers import BertConfig, BertModel

torch.manual_seed(0)
cfg = BertConfig(vocab_size=64, hidden_size=32, num_hidden_layers=2,
                 num_attention_heads=2, intermediate_size=64,
                 max_position_embeddings=32)
model = BertModel(cfg).eval()
ids = torch.randint(0, cfg.vocab_size, (2, 8))
mask = torch.ones_like(ids)
with torch.no_grad():
    out = model(input_ids=ids, attention_mask=mask)
n_params = sum(p.numel() for p in model.parameters())
assert out.last_hidden_state.shape == (2, 8, 32), out.last_hidden_state.shape
assert out.pooler_output.shape == (2, 32)
print(f"BertModel forward OK — last_hidden_state={tuple(out.last_hidden_state.shape)}, {n_params:,} params")

In [ ]:
# Example 3 — training runtime: a BertForMaskedLM forward WITH labels yields a
# loss; one backward + optimizer step must produce finite grads and move weights.
import torch
from transformers import BertConfig, BertForMaskedLM

torch.manual_seed(0)
cfg = BertConfig(vocab_size=64, hidden_size=32, num_hidden_layers=2,
                 num_attention_heads=2, intermediate_size=64, max_position_embeddings=32)
mlm = BertForMaskedLM(cfg).train()
opt = torch.optim.SGD(mlm.parameters(), lr=0.5)
ids = torch.randint(0, cfg.vocab_size, (2, 8))
labels = torch.randint(0, cfg.vocab_size, (2, 8))
w0 = mlm.bert.embeddings.word_embeddings.weight.detach().clone()
out = mlm(input_ids=ids, labels=labels)
out.loss.backward()
grad_norm = sum(p.grad.abs().sum().item() for p in mlm.parameters() if p.grad is not None)
opt.step()
assert out.loss.requires_grad and torch.isfinite(out.loss)
assert grad_norm > 0
assert not torch.equal(w0, mlm.bert.embeddings.word_embeddings.weight.detach())
print(f"MaskedLM train step OK — loss={out.loss.item():.4f}, grad_norm={grad_norm:.3f}, weights updated")

In [ ]:
# Example 4 — serialization runtime + weight-identity invariant: save the model
# to a temp dir (safetensors, native) and reload; a fresh forward must match the
# original bit-for-bit. Catches broken safetensors serialization.
import tempfile, os, torch
from transformers import BertConfig, BertModel

torch.manual_seed(0)
cfg = BertConfig(vocab_size=64, hidden_size=32, num_hidden_layers=2,
                 num_attention_heads=2, intermediate_size=64, max_position_embeddings=32)
model = BertModel(cfg).eval()
ids = torch.randint(0, cfg.vocab_size, (1, 6))
with torch.no_grad():
    ref = model(ids).last_hidden_state
with tempfile.TemporaryDirectory() as d:
    model.save_pretrained(d)
    files = sorted(os.listdir(d))
    reloaded = BertModel.from_pretrained(d).eval()
    with torch.no_grad():
        got = reloaded(ids).last_hidden_state
assert "model.safetensors" in files, files
assert torch.equal(ref, got), (ref - got).abs().max()
print(f"save/load round-trip OK — wrote {files}, reload matches exactly (maxdiff=0)")

In [ ]:
# Example 5 — generation runtime on device. Autoregressive greedy decode on a tiny
# GPT-2 (CPU), then the same model + input moved to CUDA and generated on-device.
import torch
from transformers import GPT2Config, GPT2LMHeadModel

torch.manual_seed(0)
cfg = GPT2Config(vocab_size=64, n_positions=32, n_embd=32, n_layer=2, n_head=2,
                 bos_token_id=None, eos_token_id=None)
gpt = GPT2LMHeadModel(cfg).eval()
prompt = torch.randint(0, cfg.vocab_size, (1, 4))
gen = gpt.generate(prompt, max_new_tokens=8, do_sample=False)
assert gen.shape == (1, 12), gen.shape
print(f"CPU generate OK — {prompt.shape[1]} -> {gen.shape[1]} tokens (greedy)")

assert torch.cuda.is_available(), (
    f"requires-gpu cell ran but torch.cuda.is_available() is False — torch {torch.__version__}, "
    f"built against CUDA {torch.version.cuda}, device_count={torch.cuda.device_count()}"
)

gpt_cuda = gpt.cuda()
gen_cuda = gpt_cuda.generate(prompt.cuda(), max_new_tokens=8, do_sample=False)
torch.cuda.synchronize()
assert gen_cuda.is_cuda and tuple(gen_cuda.shape) == (1, 12)
print(f"GPU generate OK on {gen_cuda.device} — output {tuple(gen_cuda.shape)}")